In [8]:
from Bio import SeqIO
import pandas as pd

In [27]:
# Parse the contig alignment file
contigs = '/home/tobamo/analize/project-tobamo/analysis/phylogenetic_placement/results/untrimmed/placement/contigs_aligned.fasta'

for record in SeqIO.parse(contigs, "fasta"):
    print(len(record.seq))
    break

9783


In [28]:
def extract_coordinates(fasta_file, output_file=None):
    """
    Parse aligned FASTA and extract coordinates of non-gap regions.
    
    Args:
        fasta_file: Path to input FASTA file
        output_file: Path to output file (if None, prints to stdout)
    """
    results = []  # Use a list instead
    
    with open(fasta_file, 'r') as f:
        current_header = None
        current_seq = []
        
        for line in f:
            line = line.rstrip('\n')
            
            if line.startswith('>'):
                # Process previous sequence if exists
                if current_header is not None:
                    start, end = find_gap_boundaries(''.join(current_seq))
                    if start is not None:
                        results.append({'contig_name': current_header, 'start': start, 'end': end})
                
                # Start new sequence
                current_header = line[1:].lower()
                current_seq = []
            else:
                current_seq.append(line)
        
        # Process last sequence
        if current_header is not None:
            start, end = find_gap_boundaries(''.join(current_seq))
            if start is not None:
                results.append({'contig_name': current_header, 'start': start, 'end': end})
    
    # Convert to DataFrame
    df_results = pd.DataFrame(results)
    df_results.rename(columns={'contig_name': 'contig_name', 'start': 'start[0]', 'end': 'end[9783]'}, inplace=True)
    
    # Output results
    if output_file:
        df_results.to_csv(output_file, index=False)  # Better for tabular output
    
    return df_results

def find_gap_boundaries(sequence):
    """
    Find the position of first and last non-gap character (1-indexed).
    
    Args:
        sequence: DNA sequence string (may contain '-' for gaps)
        
    Returns:
        Tuple of (start_pos, end_pos) or (None, None) if all gaps
    """
    # Find first non-gap position
    start_pos = None
    for i, char in enumerate(sequence):
        if char != '-':
            start_pos = i + 1  # Convert to 1-indexed
            break
    
    # Find last non-gap position
    end_pos = None
    for i in range(len(sequence) - 1, -1, -1):
        if sequence[i] != '-':
            end_pos = i + 1  # Convert to 1-indexed
            break
    
    return start_pos, end_pos

In [31]:
df = extract_coordinates(contigs)
df.to_excel('/home/tobamo/analize/project-tobamo/analysis/phylogenetic_placement/results/untrimmed/placement/contig_coordinates.xlsx', index=False)